# Task 3 — Data Augmentation

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Integration & Augmentation**

---

Augment existing **DC charger records** with new information from Open Charge Map (OCM) and saved PlugShare results. The notebook reads `data/interim/ev_chargers_clean.csv` from Task 2 and should be run from top to bottom. Cached source responses are reused when available.

## Workflow

1. [OCM augmentation](#1-ocm-augmentation)
   - [Load and validate OCM data](#1-2-read-or-retrieve-the-ocm-snapshot)
   - [Clean addresses and operators](#1-3-normalise-text-and-flatten-ocm-records)
   - [Match candidates and record decisions](#1-6-apply-the-matching-rules)
   - [Deduplicate shared OCM sites](#1-10-review-shared-ocm-sites)

2. [PlugShare augmentation](#2-plugshare-augmentation)
   - [Load saved searches](#2-3-load-saved-plugshare-searches)
   - [Check DC evidence and match candidates](#2-4-inspect-plugshare-dc-evidence)
   - [Record accepted and rejected matches](#2-6-resolve-plugshare-candidates)

3. [Final dataset](#3-final-dataset)
   - [Combine plug types](#3-2-start-with-deduplicated-ocm-records)
   - [Validate and export](#3-5-save-interim-and-final-outputs)

## Outputs

- `data/raw/`: downloaded OCM and PlugShare responses.
- `data/interim/ocm_matching_review.csv` and `plugshare_matching_review.csv`: every DC record's chosen candidate, distance, evidence and decision.
- `data/interim/ev_chargers_augmented_full.csv`: the complete audit dataset — all charger fields plus every OCM and PlugShare attribute, external ID and match status. Task 4 loads this file.
- `data/processed/ev_chargers_augmented.csv`: final cleaned dataset containing the original charger fields and one combined `plug_type` field.

The final processed file excludes matching diagnostics and source-specific IDs. Those details remain in the interim audit files so every augmentation decision can be checked.


# Imports, paths and retrieval settings

Required packages: pandas, numpy, requests, python-dotenv, duckdb and IPython.
The code finds the project folder, loads Task 2 data, and selects DC records.
`OCM_REFRESH=False` reuses a verified snapshot. Set it to `True` only to retrieve an updated snapshot.
An API key is needed only for retrieval; keep `OCM_API_KEY` in `.env`.


In [1]:
%pip install -q "requests>=2.32" "python-dotenv>=1.0" "pandas>=2.2" "numpy>=1.26" "duckdb>=1.5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import re
import math
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv

# Locate the Task 2 output. The notebook runs from the project root, like the others.
project_root = Path.cwd()
clean_input = project_root / "data/interim/ev_chargers_clean.csv"
if not clean_input.exists():
    raise FileNotFoundError(
        "data/interim/ev_chargers_clean.csv is missing - run 02_data_cleaning.ipynb first."
    )

# Preserve postcodes as text and keep the full input table for the final left join.
chargers = pd.read_csv(clean_input, dtype={"postcode": "string", "sa4_code": "string"})
assert chargers["charger_id"].notna().all() and chargers["charger_id"].is_unique

# Only DC records are augmentation targets; AC rows will still survive in the final CSV.
dc_chargers = chargers.loc[chargers["charger_type"].eq("DC")].copy()
if dc_chargers.empty:
    raise ValueError("No DC records found in the Task 2 output")

# Exclude missing or implausible coordinates from the search rectangle, not from coverage.
eligible_coords = dc_chargers["latitude"].between(-37.6, -28.1) & dc_chargers["longitude"].between(
    140.9, 153.7
)
search_chargers = dc_chargers.loc[eligible_coords]
if search_chargers.empty:
    raise ValueError("No usable NSW DC coordinates")

OCM_ENDPOINT = "https://api.openchargemap.io/v3/poi/"
OCM_MAX_RESULTS = 10000
# False: reuse verified saved JSON. True: request current data and replace the cache.
# To update, change this to True and rerun sections 1.2-1.12; then set it back to False.
OCM_REFRESH = False
MATCH_RADIUS_M = 250.0
ADDRESS_SIMILARITY_MIN = 0.70
SEARCH_MARGIN_DEG = 0.02
# API filters: request Australia within the rectangle surrounding our DC sites.
# compact=false includes descriptive objects such as CurrentType and ConnectionType.
OCM_PARAMS = {
    "output": "json",
    "countrycode": "AU",
    "maxresults": OCM_MAX_RESULTS,
    "compact": "false",
    "verbose": "false",
    "boundingbox": (
        f"({search_chargers.latitude.max() + SEARCH_MARGIN_DEG},{search_chargers.longitude.min() - SEARCH_MARGIN_DEG}),"
        f"({search_chargers.latitude.min() - SEARCH_MARGIN_DEG},{search_chargers.longitude.max() + SEARCH_MARGIN_DEG})"
    ),
}


ocm_raw_dir = project_root / "data/raw/openchargemap"
plugshare_raw_dir = project_root / "data/raw/plugshare"
interim_dir = project_root / "data/interim"
processed_dir = project_root / "data/processed"
interim_dir.mkdir(parents=True, exist_ok=True)
ocm_raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
# Use readable filenames. Saved metadata checks whether the query filters changed.
# A refresh replaces this snapshot instead of creating another set of files.
ocm_cache = ocm_raw_dir / "locations.json"
ocm_metadata = ocm_raw_dir / "locations.metadata.json"
print(f"{len(dc_chargers)} DC records; target: {math.ceil(len(dc_chargers) / 2)} augmented records")
print("Search bounding box:", OCM_PARAMS["boundingbox"])
from IPython.display import display
import duckdb

for env_path in [project_root / ".env", project_root.parent / ".env"]:
    if env_path.is_file():
        load_dotenv(env_path, override=False)

# Keep notebook output readable when tables contain long lists or text.
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 45)


431 DC records; target: 216 augmented records
Search bounding box: (-28.15379,141.440104),(-36.717259000000006,153.635875)


# 1. OCM augmentation

This section retrieves OCM data, cleans comparison fields, matches candidates, measures coverage, and deduplicates shared OCM sites.


## 1.2 Read or retrieve the OCM snapshot

The request uses the rectangle covering our DC coordinates, restricted to Australia.
`compact=false` keeps descriptive operator and connector objects. The saved metadata records
query parameters, retrieval time and a SHA-256 checksum, so reruns can reuse the same input.
The API limit is checked to avoid silently treating a truncated response as complete.
Transient GET failures are retried. API keys are not saved in outputs.

API reference: https://openchargemap.org/site/develop/api


In [3]:
def validate_ocm_payload(payload):
    """Check response structure and reject a possibly truncated result set."""
    if not isinstance(payload, list) or not all(
        isinstance(p, dict) and p.get("ID") is not None for p in payload
    ):
        raise ValueError("Expected a JSON list of OCM locations with IDs")
    if len(payload) >= OCM_MAX_RESULTS:
        raise ValueError(
            "OCM result limit reached; retrieve a complete snapshot before matching"
        )
    return payload


# Try the local snapshot first unless a fresh download was explicitly requested.
ocm_meta = None
if not OCM_REFRESH and ocm_cache.exists() and ocm_metadata.exists():
    try:
        meta = json.loads(ocm_metadata.read_text(encoding="utf-8"))
        body = ocm_cache.read_bytes()
        # Compare the actual query dictionary and the saved file fingerprint.
        # This verifies local data, not whether the server has newer data.
        if (
            meta["parameters"] == OCM_PARAMS
            and hashlib.sha256(body).hexdigest() == meta["sha256"]
        ):
            ocm_payload = validate_ocm_payload(json.loads(body))
            ocm_meta = meta
            print("Using saved OCM data; set OCM_REFRESH=True to download updates.")
    except (OSError, ValueError, KeyError):
        print("Cache could not be verified; a fresh retrieval is required.")

# No reusable snapshot: authenticate, request current results, and validate before saving.
# .env files were loaded explicitly in section 1.
if ocm_meta is None:
    print("Downloading a fresh OCM snapshot.")
    api_key = os.environ.get("OCM_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("Set OCM_API_KEY in the environment or .env file ")
    with requests.Session() as ocm_session:
        retries = Retry(
            total=4,
            backoff_factor=1.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
        )
        ocm_session.mount("https://", HTTPAdapter(max_retries=retries))
        ocm_session.headers.update(
            {
                "X-API-Key": api_key,
                "User-Agent": "COMP5339-Assignment1-DataAugmentation/1.0",
            }
        )
        response = ocm_session.get(OCM_ENDPOINT, params=OCM_PARAMS, timeout=(15, 120))
        response.raise_for_status()
        body = response.content
        ocm_payload = validate_ocm_payload(response.json())
    # Save to a temporary file first so interrupted writes do not replace a valid snapshot.
    temporary = ocm_cache.with_suffix(".part")
    temporary.write_bytes(body)
    temporary.replace(ocm_cache)
    # Store enough information to identify and verify this snapshot, without saving the key.
    ocm_meta = {
        "endpoint": OCM_ENDPOINT,
        "parameters": OCM_PARAMS,
        "retrieved_at": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(body).hexdigest(),
        "records": len(ocm_payload),
    }
    temporary = ocm_metadata.with_suffix(".part")
    temporary.write_text(json.dumps(ocm_meta, indent=2), encoding="utf-8")
    temporary.replace(ocm_metadata)
    del api_key
print(f"{len(ocm_payload)} OCM records; snapshot retrieved {ocm_meta['retrieved_at']}")


Using saved OCM data; set OCM_REFRESH=True to download updates.
380 OCM records; snapshot retrieved 2026-09-20T17:53:21.915829+00:00


## 1.3 Normalise text and flatten OCM records

Matching ignores case, punctuation and common address spelling differences, such as
`Road`/`Rd` and `New South Wales`/`NSW`. Original addresses remain visible.
Each OCM site becomes one row. Only connections explicitly labelled DC contribute plug types;
AC connectors at a mixed site are not added to a DC record.


In [4]:
def clean_text(value):
    """Convert a missing value to empty text; otherwise trim spaces."""
    return "" if value is None or pd.isna(value) else str(value).strip()


def normalise_key(value):
    """Ignore case and punctuation when comparing names or addresses."""
    return re.sub(r"[^a-z0-9]+", " ", clean_text(value).lower()).strip()


def normalise_address(value):
    """Normalise comparison text only; keep the original addresses for display."""
    # Lowercase both sides, replace commas/punctuation with spaces, and trim.
    key = normalise_key(value)
    # Treat the full state name and its abbreviation as the same text.
    key = re.sub(r"\bnew south wales\b", "nsw", key)
    # PlugShare commonly appends the country; OCM may omit it.
    # Remove only a trailing country token so street text is unchanged.
    key = re.sub(r"\s+ australia$", "", key)
    for long, short in {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "highway": "hwy",
        "drive": "dr",
    }.items():
        key = re.sub(r"\b" + long + r"\b", short, key)
    return key


def extract_street_number(value):
    """Extract the first number as supporting evidence, not a full address parser."""
    match = re.search(r"\b\d+[a-z]?\b", normalise_address(value))
    return match.group() if match else ""


In [5]:
# Flatten each nested API location into one row for matching and enrichment.
ocm_rows = []
for poi in ocm_payload:
    address = poi.get("AddressInfo") or {}
    operator = poi.get("OperatorInfo") or {}
    provider = poi.get("DataProvider") or {}
    connections = poi.get("Connections") or []
    # Sites can contain both AC and DC. Keep only explicitly DC connections.
    dc_connections = []
    for connection in connections:
        current_type = connection.get("CurrentType") or {}
        if (
            clean_text(current_type.get("Title")).upper() == "DC"
            or connection.get("CurrentTypeID") == 30
        ):
            dc_connections.append(connection)
    connector_types = sorted(
        {clean_text((c.get("ConnectionType") or {}).get("Title")) for c in dc_connections}
        - {"", "Unknown", "Other"}
    )
    ocm_rows.append(
        {
            "ocm_id": poi["ID"],
            "ocm_title": address.get("Title"),
            "ocm_address": ", ".join(
                clean_text(address.get(k))
                for k in [
                    "AddressLine1",
                    "AddressLine2",
                    "Town",
                    "StateOrProvince",
                    "Postcode",
                ]
                if clean_text(address.get(k))
            ),
            "ocm_street": address.get("AddressLine1"),
            "ocm_postcode": clean_text(address.get("Postcode")),
            "ocm_latitude": address.get("Latitude"),
            "ocm_longitude": address.get("Longitude"),
            "ocm_operator": operator.get("Title"),
            "ocm_dc_confirmed": bool(dc_connections),
            "ocm_plugtype": "; ".join(connector_types),
            "ocm_usage_type": (poi.get("UsageType") or {}).get("Title"),
            "ocm_usage_cost": poi.get("UsageCost"),
            "ocm_operator_website": operator.get("WebsiteURL"),
            "ocm_date_last_verified": poi.get("DateLastVerified"),
            "ocm_data_provider": provider.get("Title"),
            "ocm_license": provider.get("License"),
            "ocm_source_url": f"https://openchargemap.org/poi/details/{poi['ID']}",
            "ocm_retrieved_at": ocm_meta["retrieved_at"],
        }
    )
if not ocm_rows:
    raise ValueError("OCM returned no locations; inspect the request before proceeding")
ocm_locations = pd.DataFrame(ocm_rows)
if ocm_locations["ocm_id"].duplicated().any():
    raise ValueError("Repeated OCM IDs in response; review the source snapshot")
for col in ["ocm_latitude", "ocm_longitude"]:
    ocm_locations[col] = pd.to_numeric(ocm_locations[col], errors="coerce")
print(
    f"{int(ocm_locations.ocm_dc_confirmed.sum())} sites have explicitly labelled DC connections"
)


272 sites have explicitly labelled DC connections


## 1.4 Compare and standardise operator names

Use Task 2's explicit lookup approach: lowercase and normalise whitespace for lookup,
then return a canonical display name. Known aliases are listed rather than guessed by fuzzy matching.
For example, `Evie` becomes `Evie Networks`, and `BP Pulse (AU)` becomes `BP Australia`.
The table shows **every OCM operator**, its converted name and corresponding original names.
Generic unknown/business-owner labels provide no operator identity evidence.
Tesla variants share an operator, but the raw labels still describe different access conditions.

After the audit, the canonical name replaces `ocm_operator` directly. Matching and output tables
show only the source `operator` and cleaned `ocm_operator`, without duplicate canonical columns.

The same `OPERATOR_ALIASES` table is reused for OCM and PlugShare.


In [6]:
# Operator audit: use Task 2's explicit lookup approach, preserving raw source labels.
TASK2_OPERATOR_CANONICAL = {
    "chargehub": "ChargeHub",
    "charge hub": "ChargeHub",
    "non-networked": "Non-networked",
    "non networked": "Non-networked",
    "bp": "BP Australia",
    "bp australia": "BP Australia",
    "tesla": "Tesla",
    "tesla motors": "Tesla",
    "nrma": "NRMA",
    "nrma electric": "NRMA",
    "evie": "Evie Networks",
    "evie networks": "Evie Networks",
    "plus es": "PLUS ES",
    "plus es manag": "PLUS ES",
    "viva energy a": "Viva Energy Australia",
    "viva energy australia": "Viva Energy Australia",
}
# One explicit alias table is shared by OCM and PlugShare.
OPERATOR_ALIASES = {
    'bp pulse (au)': 'BP Australia', 'bp pulse': 'BP Australia', 'bp pulse australia': 'BP Australia',
    'ampcharge': 'Ampol', 'ampol ampcharge': 'Ampol', 'ampol': 'Ampol',
    'evie': 'Evie Networks', 'evie networks': 'Evie Networks',
    'tesla (tesla-only charging)': 'Tesla', 'tesla (including non-tesla)': 'Tesla',
    'tesla supercharger': 'Tesla', 'tesla destination': 'Tesla', 'tesla destination charging': 'Tesla',
    'evx (au)': 'EVX', 'smart charge (au)': 'Smart Charge', 'elanga (au)': 'Elanga',
    'wevolt (au)': 'Wevolt', 'charge hub': 'ChargeHub', 'charge fox': 'Chargefox',
    'chargefox': 'Chargefox', 'nrma': 'NRMA', 'jolt': 'JOLT',
}
# Canonical spellings for names Task 2 deliberately passed through unchanged.
OCM_OPERATOR_CANONICAL = {
    re.sub(r"\s+", " ", str(name)).strip().lower(): str(name)
    for name in chargers["operator"].dropna().unique()
}
OCM_OPERATOR_CANONICAL.update(OPERATOR_ALIASES)

OCM_OPERATOR_UNKNOWN = {
    "",
    "(unknown operator)",
    "(business owner at location)",
    "(private residence/individual)",
    "unknown",
    "non-networked",
    "non networked",
    "university of",
}


def match_canonical_operator(value):
    if pd.isna(value):
        return pd.NA
    key = re.sub(r"\s+", " ", str(value)).strip().lower()
    if key in OCM_OPERATOR_UNKNOWN:
        return pd.NA
    return OCM_OPERATOR_CANONICAL.get(key, str(value).strip())


# List every OCM label alongside the original TfNSW labels with the same canonical name.
source_operator_names = (
    chargers[["operator_raw", "operator"]].drop_duplicates().copy()
)
source_operator_names["canonical"] = source_operator_names["operator"].map(
    match_canonical_operator
)
source_by_operator = (
    source_operator_names.dropna(subset=["canonical"])
    .groupby("canonical")["operator_raw"]
    .agg(lambda values: "; ".join(sorted(set(values.dropna().astype(str)))))
)
operator_audit = (
    ocm_locations.groupby("ocm_operator", dropna=False)
    .size()
    .reset_index(name="ocm_locations")
)
operator_audit["operator_canonical"] = operator_audit.ocm_operator.map(
    match_canonical_operator
)
operator_audit["original_tfnsw_names"] = operator_audit.operator_canonical.map(
    source_by_operator
)
operator_audit["comparison"] = operator_audit.apply(
    lambda row: (
        "missing operator evidence"
        if pd.isna(row.operator_canonical)
        else (
            "not in original dataset"
            if pd.isna(row.original_tfnsw_names)
            else (
                "same canonical spelling"
                if row.ocm_operator == row.operator_canonical
                else "alias / case variant converted"
            )
        )
    ),
    axis=1,
)
print("All OCM operator names compared with the original dataset:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(operator_audit)
print("Original canonical operators not represented in OCM:")
print(
    sorted(
        set(source_by_operator.index) - set(operator_audit.operator_canonical.dropna())
    )
)
# Clean the existing column in place, just as Task 2 cleans operator names.
# Original API labels remain in the raw JSON and the audit table above.
ocm_locations["ocm_operator"] = ocm_locations["ocm_operator"].map(
    match_canonical_operator
)


All OCM operator names compared with the original dataset:


,ocm_operator,ocm_locations,operator_canonical,original_tfnsw_names,comparison
0,(Business Owner at Location),8,<NA>,NaN,missing operator evidence
1,(Private Residence/Individual),1,<NA>,NaN,missing operator evidence
2,(Unknown Operator),41,<NA>,NaN,missing operator evidence
3,Ampol AmpCharge,9,Ampol,Ampol,alias / case variant converted
4,BP Pulse (AU),17,BP Australia,BP; BP Australia,alias / case variant converted
5,Blink Charging,1,Blink Charging,NaN,not in original dataset
6,ChargePoint,4,ChargePoint,ChargePoint,same canonical spelling
7,Chargefox,58,Chargefox,Chargefox,same canonical spelling
8,Chargehub,1,ChargeHub,Charge Hub; ChargeHub,alias / case variant converted
9,EO Charging,3,EO Charging,NaN,not in original dataset


Original canonical operators not represented in OCM:
['360 EV Charge', 'AXCharge', 'Alchemy Charge', 'BMW', 'CasaCharge', 'Charge OS', 'ChargePost', 'Chargestar', 'Counties Energy', 'EV Meter', 'EVE Australia', 'EVNet', 'EVSE', 'EVUp', 'Energy Austra', 'Engie', 'Fast Cities A', 'Gentari', 'Noodoe', 'PLUS ES', 'Porsche Destination Charging', 'Porsche Smart Mobility', 'Saascharge', 'Viva Energy Australia', 'Zeus Renewables']


## 1.5 Define street comparison evidence

Street numbers are optional supporting evidence. The street name must be similar
(at least **0.90**) when the distance is over 50 m; differing or missing house numbers are
recorded but do not by themselves reject a candidate. These are chosen thresholds, not probabilities of a correct match.
Postcodes are compared separately. The original and OCM address fields are never overwritten.


In [7]:
def split_street_address(value):
    """Separate a leading house number from street text; never use a postcode as a house number."""
    text = clean_text(value).split(",")[0].strip().lower()
    number_match = re.match(r"^(\d+[a-z]?(?:\s*[-/]\s*\d+[a-z]?)?)\s+", text)
    number = re.sub(r"\s+", "", number_match.group(1)) if number_match else ""
    name = text[number_match.end() :] if number_match else text
    name = normalise_address(name)
    # Remove trailing suburb/state text when a normal street suffix is present.
    # A complex address without a recognised suffix remains conservative for review.
    street = re.match(
        r"^(.+?\b(?:st|rd|ave|hwy|dr|lane|ln|pde|parade|crescent|cres|way|place|pl|court|ct|terrace|tce))\b",
        name,
    )
    return number, street.group(1) if street else name


In [8]:
def address_evidence(charger, candidate, distance):
    """Calculate address, postcode and DC evidence without making a match decision."""
    # Compare the street portion separately from suburb/state formatting.
    source_street = clean_text(charger["station_address"]).split(",")[0]
    candidate_street = clean_text(candidate["ocm_street"])
    street_similarity = SequenceMatcher(
        None, normalise_address(source_street), normalise_address(candidate_street)
    ).ratio()
    source_number, source_name = split_street_address(source_street)
    candidate_number, candidate_name = split_street_address(candidate_street)
    both_numbered = bool(source_number and candidate_number)
    street_number_match = both_numbered and source_number == candidate_number
    street_number_conflict = both_numbered and source_number != candidate_number
    street_name_similarity = (
        SequenceMatcher(None, source_name, candidate_name).ratio()
        if source_name and candidate_name
        else 0.0
    )

    address_match_method = (
        "number_and_street" if both_numbered else "street_name_missing_number"
    )
    # Keep full addresses in the review output even though street text drives this rule.
    address_similarity = SequenceMatcher(
        None,
        normalise_address(charger["station_address"]),
        normalise_address(candidate["ocm_street"]),
    ).ratio()
    source_postcode = clean_text(charger["postcode"])
    candidate_postcode = clean_text(candidate["ocm_postcode"])
    postcode_conflict = bool(
        source_postcode and candidate_postcode and source_postcode != candidate_postcode
    )

    return {
        "candidate_ocm_id": candidate["ocm_id"],
        "ocm_title": candidate["ocm_title"],
        "ocm_street": candidate["ocm_street"],
        "ocm_operator": candidate["ocm_operator"],
        "ocm_postcode": candidate["ocm_postcode"],
        "ocm_latitude": candidate["ocm_latitude"],
        "ocm_longitude": candidate["ocm_longitude"],
        "distance_m": float(distance),
        "street_number_match": street_number_match,
        "source_street_number": source_number,
        "ocm_street_number": candidate_number,
        "street_number_conflict": street_number_conflict,
        "street_name_similarity": street_name_similarity,
        "address_match_method": address_match_method,
        "street_similarity": street_similarity,
        "address_similarity": address_similarity,
        "postcode_conflict": postcode_conflict,
        "dc_confirmed": bool(candidate["ocm_dc_confirmed"]),
    }


## 1.6 Apply the matching rules

| Distance | Operator | Address | Initial decision |
|---|---|---|---|
| At most 50 m | Same/alias | Any | Accept |
| At most 50 m | Missing | Similar | Accept |
| Over 50 m, at most 250 m | Same/alias | Similar | Accept |
| At most 250 m | Different known operators or other combinations | Any | Review |
| Over 250 m | Any | Any | Reject |

**Additional conditions apply to every acceptance:**
- Known postcodes must match. Missing OCM postcode is allowed only with a present source postcode,
  same operator, similar address and distance at most 250 m. Otherwise review.
- OCM must explicitly identify a DC connection.
- Later cells select the nearest passing candidate and flag shared OCM sites for review.

Close coordinates alone do not resolve conflicting source addresses or operator identities.


In [9]:
def match_operator(value):
    return normalise_key(match_canonical_operator(value))


def match_compare(charger, candidate, distance):
    # Retain the same evidence columns for a direct before/after comparison.
    evidence = address_evidence(charger, candidate, distance)
    left, right = match_operator(charger["operator"]), match_operator(
        candidate["ocm_operator"]
    )
    unknown = {"", "unknown", "non networked", "university of", "other"}
    missing = left in unknown or right in unknown
    same = not missing and left == right
    a, aname = split_street_address(charger["station_address"])
    b, bname = split_street_address(candidate["ocm_street"])

    def number_key(text):
        return re.sub(r"\d+", lambda m: str(int(m.group())), text)

    numbered = bool(a and b)
    numbers_match = numbered and number_key(a) == number_key(b)
    # Street number is supporting evidence only. For 50–250 m matches,
    # the street name may match even when house numbers differ or are missing.
    similar = bool(aname and bname) and evidence["street_name_similarity"] >= 0.90
    if distance > 250:
        decision, reason = "reject", "distance exceeds 250 m"
    elif not missing and not same:
        decision, reason = "review", "different known operators"
    elif distance <= 50 and same:
        decision, reason = (
            "accept",
            "within 50 m and same/aliased operator; address disagreement allowed",
        )
    elif distance <= 50 and missing and similar:
        decision, reason = "accept", "within 50 m; operator missing but address similar"
    elif same and similar:
        decision, reason = (
            "accept",
            "50–250 m with same/aliased operator and similar street name; house number optional",
        )
    else:
        decision, reason = (
            "review",
            "insufficient operator/address agreement for this distance",
        )
    matrix_decision = decision
    # Missing OCM postcode is allowed only with matching operator, similar address and distance <=250 m.
    source_postcode = clean_text(charger["postcode"])
    external_postcode = clean_text(candidate["ocm_postcode"])
    postcode_match = bool(
        source_postcode and external_postcode and source_postcode == external_postcode
    )
    evidence["postcode_match"] = postcode_match
    postcode_missing_exception = bool(
        source_postcode
        and not external_postcode
        and same
        and similar
        and distance <= 250
    )
    evidence["postcode_missing_exception"] = postcode_missing_exception
    if decision == "accept" and postcode_missing_exception:
        reason += "; OCM postcode missing; operator, address and distance support match"
    if decision == "accept" and not (postcode_match or postcode_missing_exception):
        decision = "review"
        reason += "; " + (
            "postcode missing on one or both sides"
            if not source_postcode or not external_postcode
            else "postcodes disagree"
        )
    if decision == "accept" and not evidence["dc_confirmed"]:
        decision, reason = "review", reason + "; OCM DC connection not confirmed"
    evidence.update(
        operator_match=same,
        operator_relation=(
            "missing" if missing else "same/alias" if same else "different"
        ),
        street_number_match=numbers_match,
        street_number_conflict=numbered and not numbers_match,
        address_similar=similar,
        matrix_decision=matrix_decision,
        rule_decision=decision,
        street_number_policy="optional_supporting_evidence",
        candidate_passes=decision == "accept",
        candidate_reason=reason,
        ocm_address=candidate["ocm_address"],
    )
    return evidence


## 1.7 Calculate candidate distances with DuckDB Spatial

Compare each DC record against OCM coordinates using WGS84 spheroidal distance in **metres**.
This DuckDB function expects `POINT_2D(latitude, longitude)` inputs.
Keep all candidates within 250 m, plus the nearest candidate for unmatched inspection.
Ordering by OCM ID breaks equal-distance display ties; it does not establish a match.


In [10]:
valid_ocm = ocm_locations.loc[
    ocm_locations.ocm_latitude.between(-90, 90)
    & ocm_locations.ocm_longitude.between(-180, 180)
].reset_index(drop=True)

# DuckDB Spatial builds the candidate list and measures distances in metres.
# The DataFrames are registered as SQL views; no extra database file is needed.
import duckdb

with duckdb.connect() as spatial_con:
    try:
        spatial_con.execute("LOAD spatial")
    except duckdb.Error:
        # Only install if not already available (installation needs network access).
        spatial_con.execute("INSTALL spatial")
        spatial_con.execute("LOAD spatial")

    spatial_con.register("dc_input", dc_chargers)
    spatial_con.register("ocm_input", valid_ocm)

    # ST_Distance_Spheroid takes WGS84 POINT_2D values in LATITUDE, LONGITUDE
    # order. This differs from the longitude/latitude geometries used in Task 2.
    # It measures on the WGS84 ellipsoid, unlike ST_Distance on degree coordinates.
    # SQL first measures each pair, then ranks candidates separately for each charger.
    # Keep all candidates within the radius AND the nearest site for unmatched review.
    # At this dataset size (~431 x 380) evaluating all pairs is inexpensive.
    spatial_candidates = spatial_con.execute(
        """
        WITH dc_points AS (
            SELECT charger_id,
                   ST_Point(latitude, longitude)::POINT_2D AS point
            FROM dc_input
            WHERE latitude BETWEEN -37.6 AND -28.1
              AND longitude BETWEEN 140.9 AND 153.7
        ), ocm_points AS (
            SELECT ocm_id,
                   ST_Point(ocm_latitude, ocm_longitude)::POINT_2D AS point
            FROM ocm_input
        ), distances AS (
            SELECT d.charger_id, o.ocm_id,
                   ST_Distance_Spheroid(d.point, o.point) AS distance_m
            FROM dc_points AS d
            CROSS JOIN ocm_points AS o
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY charger_id ORDER BY distance_m, ocm_id
            ) AS distance_rank
            FROM distances
        )
        SELECT charger_id, ocm_id, distance_m, distance_rank
        FROM ranked
        WHERE distance_m <= ? OR distance_rank = 1
        ORDER BY charger_id, distance_m, ocm_id
    """,
        [MATCH_RADIUS_M],
    ).df()


## 1.8 Choose a candidate for each charger

Apply the rules to every candidate. If several pass, select the nearest; exact distance ties
use the external ID for a deterministic choice. Record the passing-candidate count and reason. If none pass, retain the nearest candidate and its failure reason.
Every original DC record receives a review row, including records with unusable coordinates.


In [11]:
# Look up the SQL results by charger ID, then apply the existing text checks.
spatial_by_charger = {
    charger_id: group.to_dict("records")
    for charger_id, group in spatial_candidates.groupby("charger_id")
}
ocm_by_id = valid_ocm.set_index("ocm_id", drop=False)

match_rows = []
review_rows = []
candidate_rows = []
for charger in dc_chargers.to_dict("records"):
    # One review row per source charger; source and candidate fields stay distinct.
    review = {
        "charger_id": charger["charger_id"],
        "station_name": charger["station_name"],
        "station_address": charger["station_address"],
        "operator": charger["operator"],
        "postcode": charger["postcode"],
        "latitude": charger["latitude"],
        "longitude": charger["longitude"],
        "match_status": "unmatched",
        "match_reason": "no OCM locations with usable coordinates",
        "candidate_role": "none",
    }
    match = {
        "charger_id": charger["charger_id"],
        "match_status": "unmatched",
        "ocm_id": None,
        "match_distance_m": None,
    }
    lat, lon = charger["latitude"], charger["longitude"]
    if not (
        pd.notna(lat)
        and pd.notna(lon)
        and -37.6 <= lat <= -28.1
        and 140.9 <= lon <= 153.7
    ):
        review.update(
            match_status="invalid_coordinates",
            match_reason="source coordinates missing or outside NSW range",
        )
        match["match_status"] = review["match_status"]
    elif charger["charger_id"] in spatial_by_charger:
        evidence = []
        for spatial_candidate in spatial_by_charger[charger["charger_id"]]:
            candidate = ocm_by_id.loc[spatial_candidate["ocm_id"]]
            item = match_compare(charger, candidate, spatial_candidate["distance_m"])
            evidence.append(item)
            candidate_rows.append({"charger_id": charger["charger_id"], **item})
        qualifying = [item for item in evidence if item["candidate_passes"]]

        if qualifying:
            # Choose the nearest passing candidate; ID makes exact ties deterministic.
            chosen = min(
                qualifying,
                key=lambda item: (item["distance_m"], item["candidate_ocm_id"]),
            )
            review["passing_candidates"] = len(qualifying)
            review.update(chosen)
            review.update(
                match_status="accepted",
                match_reason=chosen["candidate_reason"]
                + (
                    f"; nearest of {len(qualifying)} passing candidates"
                    if len(qualifying) > 1
                    else ""
                ),
                candidate_role="accepted_match",
            )
            match.update(
                match_status="accepted",
                ocm_id=chosen["candidate_ocm_id"],
                match_distance_m=chosen["distance_m"],
            )
        else:
            # No passing candidates: display nearest with its failure reason.
            chosen = min(
                evidence,
                key=lambda item: (item["distance_m"], item["candidate_ocm_id"]),
            )
            review.update(chosen)
            review["passing_candidates"] = 0
            review["candidate_role"] = "review_only_not_accepted"
            review["match_reason"] = chosen["candidate_reason"]
            review["match_status"] = chosen["rule_decision"]
            match["match_status"] = review["match_status"]
    match_rows.append(match)
    review_rows.append(review)

match_matches = pd.DataFrame(match_rows)
match_review = pd.DataFrame(review_rows)
match_candidates = pd.DataFrame(candidate_rows)


### OCM match preview
Show five accepted and five non-accepted records for a quick check; the full review remains in the CSV.


In [12]:
preview_columns = [
    "charger_id", "station_address", "operator",
    "ocm_address", "ocm_operator", "distance_m",
    "postcode_match", "match_status", "match_reason",
]
accepted_preview = match_review.loc[
    match_review.match_status.eq("accepted"), preview_columns
].head(5)
unmatched_preview = match_review.loc[
    ~match_review.match_status.eq("accepted"), preview_columns
].head(5)
print("Accepted OCM matches (first 5):")
display(accepted_preview.round({"distance_m": 1}))
print("Unmatched/review OCM records (first 5):")
display(unmatched_preview.round({"distance_m": 1}))


Accepted OCM matches (first 5):


,charger_id,station_address,operator,ocm_address,ocm_operator,distance_m,postcode_match,match_status,match_reason
7,24,"1 Ingham Dr, Casula NSW 2170",Evie Networks,"1 Ingham Dr, Casula, New South Wales, 2170",Evie Networks,125.4,True,accepted,50–250 m with same/aliased operator and s...
9,36,"1 Park St, Sydney, 2103",JOLT,"1 Park St, Mona Vale, NSW, 2103",JOLT,5.7,True,accepted,within 50 m and same/aliased operator; ad...
16,60,"10 Winery Dr, Port Macquarie, 2444",Tesla,"764 Pacific Highway, Port Macquarie, NSW,...",Tesla,0.1,True,accepted,within 50 m and same/aliased operator; ad...
19,69,"1067 Oxley Hwy, Thrumster, 2444",Chargefox,"1063 Oxley Highway, Thrumster, NSW, 2444",Chargefox,29.9,True,accepted,within 50 m and same/aliased operator; ad...
23,80,"11 Ewingsdale Rd, Ewingsdale, 2481",NRMA,"Ewingsdale Road, Ewingsdale, New South Wales",NRMA,47.9,False,accepted,within 50 m and same/aliased operator; ad...


Unmatched/review OCM records (first 5):


,charger_id,station_address,operator,ocm_address,ocm_operator,distance_m,postcode_match,match_status,match_reason
0,2,"01 Wallgrove Road, Sydney, 2766",BP Australia,"BP Truckstop, Wallgrove Road & Old Wallgr...",BP Australia,65.2,True,review,insufficient operator/address agreement f...
1,3,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,"82 Marsh Street, Armidale, New South Wale...",NRMA,29.2,False,review,within 50 m and same/aliased operator; ad...
2,6,"1 Bay St, Sydney, 2037",Tesla,"1 Bay Street, Broadway, NSW, 2007",Tesla,19.5,False,review,within 50 m and same/aliased operator; ad...
3,7,"1 Bells Blvd, Kingscliff, 2487",Evie Networks,"45 Cabarita Road, Bogangar, New South Wal...",NRMA,6196.5,False,reject,distance exceeds 250 m
4,12,"1 Dalgal Way, Sydney, 2037",Tesla,"1 Bay Street, Broadway, 2007",Tesla,1673.9,False,reject,distance exceeds 250 m


## 1.9 Inspect matched and unmatched records

Read both addresses and operators alongside distance, postcode evidence and the decision reason.
These are automated rule decisions, not independent verification of physical station identity.
The same information is saved in `data/interim/ocm_matching_review.csv`.


## 1.10 Review shared OCM sites

After a match is selected for each source record, a separate cross-record check asks whether several
source records selected the same external location. It does not change the individual distance,
operator, address, postcode or DC checks.

A shared ID means two source rows point to one external station. That may be the same charger
recorded twice in the TfNSW data, or several chargers at one physical site; the external ID alone
cannot tell which. Supporting evidence is the same or a similar street name, the same operator, the
same postcode and nearby coordinates.

For OCM, the check runs in section 2.1, before PlugShare matching. One representative of each shared
OCM site is kept and the others are excluded from the final dataset; the table there lists every
decision. Nothing is deleted from Task 2's cleaned file. Shared PlugShare locations are handled
differently (section 2.6).

In [13]:
#preview of records that still need PlugShare matching.
unmatched = match_review.loc[~match_review.match_status.eq("accepted")]
print(f"Unmatched/review records: {len(unmatched)}")
display(unmatched.head(5))


Unmatched/review records: 311


,charger_id,station_name,station_address,operator,postcode,latitude,longitude,match_status,match_reason,candidate_role,candidate_ocm_id,ocm_title,ocm_street,ocm_operator,ocm_postcode,...,address_similarity,postcode_conflict,dc_confirmed,postcode_match,postcode_missing_exception,operator_match,operator_relation,address_similar,matrix_decision,rule_decision,street_number_policy,candidate_passes,candidate_reason,ocm_address,passing_candidates
0,2,NaN,"01 Wallgrove Road, Sydney, 2766",BP Australia,2766,-33.811004,150.849597,review,insufficient operator/address agreement f...,review_only_not_accepted,272521,BP Pulse Eastern Creek,BP Truckstop,BP Australia,2766,...,0.153846,False,True,True,False,True,same/alias,False,review,review,optional_supporting_evidence,False,insufficient operator/address agreement f...,"BP Truckstop, Wallgrove Road & Old Wallgr...",0
1,3,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,2836,-30.511874,151.669395,review,within 50 m and same/aliased operator; ad...,review_only_not_accepted,170824,Armidale Visitors Centre,82 Marsh Street,NRMA,2350,...,0.235294,True,True,False,False,True,same/alias,False,accept,review,optional_supporting_evidence,False,within 50 m and same/aliased operator; ad...,"82 Marsh Street, Armidale, New South Wale...",0
2,6,NaN,"1 Bay St, Sydney, 2037",Tesla,2037,-33.883504,151.194433,review,within 50 m and same/aliased operator; ad...,review_only_not_accepted,311308,Tesla Destination Charger,1 Bay Street,Tesla,2007,...,0.571429,True,False,False,False,True,same/alias,True,accept,review,optional_supporting_evidence,False,within 50 m and same/aliased operator; ad...,"1 Bay Street, Broadway, NSW, 2007",0
3,7,NaN,"1 Bells Blvd, Kingscliff, 2487",Evie Networks,2487,-28.276903,153.577078,reject,distance exceeds 250 m,review_only_not_accepted,295994,NRMA Cabarita Beach,45 Cabarita Road,NRMA,2488,...,0.190476,True,True,False,False,False,different,False,reject,reject,optional_supporting_evidence,False,distance exceeds 250 m,"45 Cabarita Road, Bogangar, New South Wal...",0
4,12,NaN,"1 Dalgal Way, Sydney, 2037",Tesla,2037,-33.876501,151.178000,reject,distance exceeds 250 m,review_only_not_accepted,100404,Broadway Supercharger,1 Bay Street,Tesla,2007,...,0.375000,True,True,False,False,True,same/alias,False,reject,reject,optional_supporting_evidence,False,distance exceeds 250 m,"1 Bay Street, Broadway, 2007",0


## 1.11 Measure OCM augmentation coverage

Only accepted OCM IDs are joined to source charger IDs. New attributes include **DC plug types**,
usage/access information, pricing text and operator website when present. A match without any
usable new attribute does not count as augmented. AC and unmatched rows remain in the final table.

Coverage uses every original DC record as the denominator. OCM alone covers well under the
50% target, so section 2 adds PlugShare for the DC records OCM did not match. No bay count is inferred from connector counts.


In [14]:
# Only new, usable facts count towards the 50% target; an external ID alone does not.
NEW_ATTRIBUTES = [
    "ocm_plugtype",
    "ocm_usage_type",
    "ocm_usage_cost",
    "ocm_operator_website",
]
ACCEPTED_STATUSES = {"accepted"}
# Attach external facts using the accepted OCM ID, not by row position.
augmentation = match_matches.merge(
    ocm_locations, on="ocm_id", how="left", validate="many_to_one"
)


def usable(value):
    return normalise_key(value) not in {
        "",
        "unknown",
        "not known",
        "not specified",
        "n a",
        "na",
        "none",
        "other",
    }


augmentation["augmented"] = augmentation.match_status.isin(
    ACCEPTED_STATUSES
) & augmentation[NEW_ATTRIBUTES].apply(lambda col: col.map(usable)).any(axis=1)
# Left join keeps every original charger, including AC and unmatched DC records.
augmented = chargers.merge(
    augmentation, on="charger_id", how="left", validate="one_to_one"
)
augmented["match_status"] = augmented["match_status"].fillna("not_targeted")
augmented["augmented"] = (
    augmented["augmented"].astype("boolean").fillna(False).astype(bool)
)
# Stop if enrichment changes the original record set or original values.
assert len(augmented) == len(chargers)
assert augmented.charger_id.is_unique
assert set(augmented.charger_id) == set(chargers.charger_id)
assert not augmented.loc[~augmented.charger_type.eq("DC"), "augmented"].any()
pd.testing.assert_frame_equal(
    augmented[chargers.columns], chargers, check_dtype=False
)



In [15]:
# Coverage denominator includes all original DC records, including unresolved ones.
augmented_count = int(augmentation.augmented.sum())

coverage = 100 * augmented_count / len(dc_chargers)
print(f'Accepted matches: {int(match_matches.match_status.eq("accepted").sum())}')
print(f"Augmented: {augmented_count}/{len(dc_chargers)} DC records ({coverage:.2f}%)")
print(
    f"50% target: {math.ceil(len(dc_chargers) / 2)} records; remaining: {max(0, math.ceil(len(dc_chargers) / 2) - augmented_count)}"
)
display(
    pd.DataFrame(
        {
            "attribute": NEW_ATTRIBUTES,
            "records": [
                int(
                    (
                        augmentation[c].map(usable)
                        & augmentation.match_status.eq("accepted")
                    ).sum()
                )
                for c in NEW_ATTRIBUTES
            ],
        }
    )
)


Accepted matches: 120
Augmented: 120/431 DC records (27.84%)
50% target: 216 records; remaining: 96


,attribute,records
0,ocm_plugtype,120
1,ocm_usage_type,120
2,ocm_usage_cost,97
3,ocm_operator_website,120


## 1.12 Validate OCM results

Check that every accepted link has valid distance, DC and postcode evidence. The full OCM review
(every DC record, its chosen candidate and the reason for the decision) is written to
`data/interim/ocm_matching_review.csv`, with raw OCM IDs, provider and licence information,
and retrieval timestamps for provenance. Shared OCM IDs are checked after deduplication, in
section 2.1.

The processed output uses one combined `plug_type` column. Source-specific IDs and match
diagnostics remain only in the interim audit files.

In [16]:
accepted = match_review.loc[match_review.match_status.eq("accepted")]
assert accepted.distance_m.le(250).all()
assert accepted.dc_confirmed.all()
assert (accepted.postcode_match | accepted.postcode_missing_exception).all()
assert not accepted.postcode_conflict.any()
# Shared OCM IDs are checked after deduplication below.
# Full OCM review (every DC record, its chosen candidate and the decision reason);
# Task 4 loads the match provenance from this file.
match_review.to_csv(interim_dir / "ocm_matching_review.csv", index=False)
print("Saved:", (interim_dir / "ocm_matching_review.csv").relative_to(project_root))


Saved: data/interim/ocm_matching_review.csv


# 2. PlugShare augmentation

This section retrieves or loads PlugShare results, checks DC evidence, matches stations, and records review decisions.


## 2.1 Prepare PlugShare searches

This step runs immediately after OCM matching, before PlugShare matching. If several source records
selected the same accepted OCM site, it displays the group and keeps one representative. The record
with a street number is preferred; ties use the most non-null source fields, then the smallest
`charger_id`. No source row is deleted from the original cleaned dataset.

The action table lists any change the saved PlugShare JSON would need: a search containing only a
removed source ID is dropped, and a removed ID is trimmed from a search shared with a kept one. The
filter is applied in memory before PlugShare matching, and the raw JSON is never modified. The table
is empty in this run, because no removed record was ever searched on PlugShare.


In [17]:
accepted_ocm = match_review.loc[
    match_review.match_status.isin(["accepted", "shared_site_review"])
    & match_review.candidate_ocm_id.notna()
].copy()
shared_ids = accepted_ocm.candidate_ocm_id.value_counts()
shared_ids = set(shared_ids[shared_ids > 1].index)
shared_ocm = accepted_ocm.loc[accepted_ocm.candidate_ocm_id.isin(shared_ids)].copy()
if shared_ocm.empty:
    dedup_keep_ids, dedup_remove_ids = set(chargers.charger_id), set()
    print("No shared accepted OCM sites found.")
else:
    shared_ocm["has_street_number"] = shared_ocm.station_address.map(
        lambda value: bool(re.match(r"^\s*\d+", clean_text(value)))
    ).astype(int)
    shared_ocm["source_non_null"] = (
        shared_ocm[[c for c in chargers.columns if c in shared_ocm.columns]]
        .notna()
        .sum(axis=1)
    )
    keep = shared_ocm.sort_values(
        ["candidate_ocm_id", "has_street_number", "source_non_null", "charger_id"],
        ascending=[True, False, False, True],
    ).drop_duplicates("candidate_ocm_id")
    dedup_keep_ids = set(keep.charger_id.astype(int)) | set(
        chargers.charger_id
    ) - set(shared_ocm.charger_id.astype(int))
    dedup_remove_ids = set(shared_ocm.charger_id.astype(int)) - set(
        keep.charger_id.astype(int)
    )
    print("Keep:", sorted(dedup_keep_ids & set(shared_ocm.charger_id.astype(int))))
    print("Remove:", sorted(dedup_remove_ids))

    # Show every shared duplicate and mark the row retained by the rule.
    shared_display = shared_ocm[[
        "candidate_ocm_id", "charger_id", "station_address", "operator",
        "ocm_address", "distance_m", "has_street_number", "source_non_null"
    ]].copy()
    shared_display["decision"] = shared_display.charger_id.astype(int).map(
        lambda value: "KEEP" if value in dedup_keep_ids else "REMOVE"
    )
    display(shared_display.sort_values(["candidate_ocm_id", "decision", "charger_id"]))

ps_path = plugshare_raw_dir / "coordinate_searches.json"
if ps_path.exists():
    ps_for_actions = json.loads(ps_path.read_text(encoding="utf-8"))
    actions = []
    for key, entry in ps_for_actions.get("searches", {}).items():
        ids = set(map(int, entry.get("charger_ids", [])))
        removed = sorted(ids & dedup_remove_ids)
        kept = sorted(ids - dedup_remove_ids)
        if removed:
            actions.append(
                {
                    "search_key": key,
                    "address": entry.get("address"),
                    "action": "remove_search" if not kept else "trim_removed_ids",
                    "removed_charger_ids": removed,
                    "kept_charger_ids": kept,
                }
            )
    plugshare_actions = pd.DataFrame(actions)
    display(plugshare_actions)
    print("PlugShare search actions are informational; the raw JSON is not modified.")
else:
    plugshare_actions = pd.DataFrame()
    print("PlugShare JSON does not exist yet; no search actions to list.")


Keep: [141, 187, 307, 409, 872, 1691]
Remove: [142, 410, 821, 829, 873, 1074]


,candidate_ocm_id,charger_id,station_address,operator,ocm_address,distance_m,has_street_number,source_non_null,decision
145,126130,409,"29-55 Twynam St, Narrandera, 2700",NRMA,"31 Twynam Street, Narrandera, NSW, 2700",16.951842,1,6,KEEP
146,126130,410,"29-55 Twynam St, Narrandera, 2700",NRMA,"31 Twynam Street, Narrandera, NSW, 2700",204.117967,1,6,REMOVE
46,190670,141,"134 Lachlan St, Hay, 2711",NRMA,"134 Lachlan St, Hay, NSW, 2711",27.349568,1,6,KEEP
47,190670,142,"134 Lachlan St, Hay, 2711",NRMA,"134 Lachlan St, Hay, NSW, 2711",174.436644,1,6,REMOVE
105,266501,307,20-22 Camden Rd Campbelltown NSW 2560 Aus...,Tesla,"20-22 Camden Road, Campbelltown, New Sout...",130.761830,1,6,KEEP
364,266501,1074,20-22 Camden Rd Campbelltown NSW 2560 Aus...,Tesla,"20-22 Camden Road, Campbelltown, New Sout...",88.191566,1,6,REMOVE
397,272619,1691,801-899R Bunnerong Rd Chifley NSW 2036 Au...,Chargefox,"439 Bunnerong Road, Chifley, New South Wa...",86.731791,1,6,KEEP
290,272619,821,"Bunnerong Rd, Sydney, 2036",Chargefox,"439 Bunnerong Road, Chifley, New South Wa...",86.723864,0,6,REMOVE
314,273428,872,"Hassall St, Sydney, 2142",BP Australia,"Hassall Street, Rosehill, New South Wales...",76.120257,0,6,KEEP
315,273428,873,"Hassall Street & James Rouse Drive, Sydne...",BP Australia,"Hassall Street, Rosehill, New South Wales...",85.382865,0,6,REMOVE


""


PlugShare search actions are informational; the raw JSON is not modified.


## 2.2 PlugShare retrieval cache

Retrieval is handled by `tools/Plug_share.py`. This notebook does not download again when
its JSON cache exists; it displays the saved searches and then continues with matching. If the cache
is missing, the next cell runs that script, which needs an Apify token. For each DC record OCM did
not match, the script searches a 1 km radius around its source coordinates and keeps up to two
results.

In [18]:
plugshare_cache_path = plugshare_raw_dir / 'coordinate_searches.json'
if plugshare_cache_path.exists():
    ps_cache = json.loads(plugshare_cache_path.read_text(encoding='utf-8'))
    print(f'PlugShare JSON found: {plugshare_cache_path.relative_to(project_root)}')
    print(f"Saved searches: {len(ps_cache.get('searches', {}))}")
    display(pd.DataFrame([
        {'address': entry.get('address'), 'charger_ids': entry.get('charger_ids'),
         'status': entry.get('status'), 'results': len(entry.get('items', []))}
        for entry in ps_cache.get('searches', {}).values()
    ]))
else:
    # Retrieve automatically through the reusable Python module.
    import sys
    sys.path.insert(0, str(project_root))
    from tools import Plug_share
    Plug_share.main()
    ps_cache = json.loads(plugshare_cache_path.read_text(encoding='utf-8'))


PlugShare JSON found: data/raw/plugshare/coordinate_searches.json
Saved searches: 311


,address,charger_ids,status,results
0,"01 Wallgrove Road, Sydney, 2766",[2],SUCCEEDED,1
1,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",[3],SUCCEEDED,2
2,"1 Bay St, Sydney, 2037",[6],SUCCEEDED,2
3,"1 Bells Blvd, Kingscliff, 2487",[7],SUCCEEDED,2
4,"1 Dalgal Way, Sydney, 2037",[12],SUCCEEDED,2
...,...,...,...,...
306,"Waddells Ave, Singleton, 2330",[978],SUCCEEDED,2
307,"Western Motorway, Sydney, 2766",[981],SUCCEEDED,2
308,"William St, Sydney, 2046",[982],SUCCEEDED,2
309,"charger 973: University of Wollongong, No...",[973],SUCCEEDED,2


### Check the cache against the current targets

The saved searches are valid only if they were made for exactly the DC records that remain
unmatched after OCM. The next cell compares the two sets: both the stale and the missing lists must
be empty. The cell after it shows any stale entries, with the reason they no longer need PlugShare.

In [19]:
# Compare cached PlugShare charger IDs with the current OCM-unmatched target set.
cache_ids = {int(cid) for entry in ps_cache.get("searches", {}).values() for cid in entry.get("charger_ids", [])}
current_unmatched_ids = set(match_review.loc[~match_review.match_status.eq("accepted"), "charger_id"].astype(int)) - dedup_remove_ids
stale_ids = sorted(cache_ids - current_unmatched_ids)
missing_ids = sorted(current_unmatched_ids - cache_ids)
print(f"Cached IDs: {len(cache_ids)} | Current unmatched IDs: {len(current_unmatched_ids)}")
print(f"Stale cached IDs: {len(stale_ids)}", stale_ids[:10])
print(f"Missing cached IDs: {len(missing_ids)}", missing_ids[:10])



Cached IDs: 311 | Current unmatched IDs: 311
Stale cached IDs: 0 []
Missing cached IDs: 0 []


In [20]:
# Show why stale cached IDs no longer need PlugShare matching.
stale_columns = ["charger_id", "station_address", "operator", "ocm_address", "ocm_operator", "distance_m", "match_status", "match_reason"]
display(match_review.loc[match_review.charger_id.isin(stale_ids), stale_columns].round({"distance_m": 1}))


,charger_id,station_address,operator,ocm_address,ocm_operator,distance_m,match_status,match_reason


## 2.3 Load saved PlugShare searches

Use only `data/raw/plugshare/coordinate_searches.json`; this section makes no API calls.
OCM accepted records keep their existing matches. A search can return several locations (`items`):
we inspect **every returned item**, not only the first. AC-only items cannot pass the DC check.


In [21]:
ps_targets = dc_chargers.loc[
    ~dc_chargers.charger_id.isin(
        set(match_matches.loc[match_matches.match_status.eq("accepted"), "charger_id"])
        | dedup_remove_ids
    )
].copy()
ps_target_ids = set(ps_targets.charger_id)
print(
    f'{len(ps_targets)} DC records remain after OCM; {len(ps_cache["searches"])} saved PlugShare searches.'
)


311 DC records remain after OCM; 311 saved PlugShare searches.


## 2.4 Inspect PlugShare DC evidence

An item qualifies as DC if at least one outlet explicitly has `isDc=true` or `powerType="DC"`.
When outlet details are absent, `isFastCharger=true` is fallback DC evidence, but no plug types
are inferred from it. Only explicitly DC outlets contribute augmentation plug types.
Operators come from DC outlets' `network.name`, with `networkNames` as a fallback.
The postcode is parsed after an Australian state abbreviation/name in the address.
Raw JSON stays unchanged. Multiple networks are retained; a source operator may match any reported DC network.

Known PlugShare labels are converted with an explicit alias table, including `AmpCharge` → `Ampol`.


In [22]:
def ps_operator(value):
    return match_canonical_operator(value)


def ps_postcode(address):
    match = re.search(
        r"\b(?:NSW|New South Wales|ACT|VIC|QLD|SA|WA|TAS|NT)\s+(\d{4})\b",
        clean_text(address),
        flags=re.I,
    )
    return match.group(1) if match else ""


ps_pairs = {}
for entry in ps_cache["searches"].values():
    if not entry.get("download_complete"):
        continue
    for item in entry.get("items", []):
        if item.get("id") is None:
            continue
        outlets = item.get("outlets") or []
        dc_outlets = [
            o
            for o in outlets
            if o.get("isDc") is True or clean_text(o.get("powerType")).upper() == "DC"
        ]
        dc_confirmed = bool(dc_outlets) or (
            not outlets and item.get("isFastCharger") is True
        )
        plugs = sorted(
            {clean_text(o.get("connectorType")) for o in dc_outlets}
            - {"", "Unknown", "Other"}
        )
        networks = [(o.get("network") or {}).get("name") for o in dc_outlets]
        if not any(networks):
            fallback = item.get("networkNames") or []
            networks = [fallback] if isinstance(fallback, str) else fallback
        operators = sorted({clean_text(ps_operator(v)) for v in networks} - {""})
        for charger_id in set(entry.get("charger_ids", [])) & ps_target_ids:
            # Deduplicate the same location returned by repeated searches for one source record.
            ps_pairs[(charger_id, str(item["id"]))] = {
                "charger_id": charger_id,
                "plugshare_id": str(item["id"]),
                "plugshare_name": item.get("name"),
                "plugshare_address": item.get("address"),
                "plugshare_latitude": item.get("latitude"),
                "plugshare_longitude": item.get("longitude"),
                "plugshare_operators": operators,
                "plugshare_postcode": ps_postcode(item.get("address")),
                "dc_confirmed": dc_confirmed,
                "dc_plug_types": "; ".join(plugs),
                "plugshare_url": item.get("plugshareUrl"),
                "retrieved_at": entry.get("downloaded_at"),
            }
ps_candidates = pd.DataFrame(
    ps_pairs.values(),
    columns=[
        "charger_id",
        "plugshare_id",
        "plugshare_name",
        "plugshare_address",
        "plugshare_latitude",
        "plugshare_longitude",
        "plugshare_operators",
        "plugshare_postcode",
        "dc_confirmed",
        "dc_plug_types",
        "plugshare_url",
        "retrieved_at",
    ],
)
for col in ["plugshare_latitude", "plugshare_longitude"]:
    ps_candidates[col] = pd.to_numeric(ps_candidates[col], errors="coerce")
print(
    f"{len(ps_candidates)} source/candidate pairs; {int(ps_candidates.dc_confirmed.sum())} have DC evidence."
)


582 source/candidate pairs; 391 have DC evidence.


## 2.5 Apply PlugShare matching rules

DuckDB measures each candidate against the original coordinates. The OCM decision function is
reused through a field adapter so the rules cannot drift between sources. A missing PlugShare
postcode follows the same exception as a missing OCM postcode. The final review uses PlugShare column names.


In [23]:
with duckdb.connect() as ps_con:
    ps_con.execute("LOAD spatial")
    ps_con.register("ps_input", ps_candidates)
    ps_con.register("source_input", ps_targets)
    ps_distances = ps_con.execute(
        """
        SELECT p.charger_id, p.plugshare_id,
               ST_Distance_Spheroid(ST_Point(s.latitude,s.longitude)::POINT_2D,
                   ST_Point(p.plugshare_latitude,p.plugshare_longitude)::POINT_2D) AS distance_m
        FROM ps_input p JOIN source_input s USING (charger_id)
        WHERE s.latitude BETWEEN -90 AND 90 AND s.longitude BETWEEN -180 AND 180
          AND p.plugshare_latitude BETWEEN -90 AND 90 AND p.plugshare_longitude BETWEEN -180 AND 180
    """
    ).df()
ps_candidates = ps_candidates.merge(
    ps_distances, on=["charger_id", "plugshare_id"], how="left", validate="one_to_one"
)
ps_source = ps_targets.set_index("charger_id")
ps_evidence = []
for row in ps_candidates.to_dict("records"):
    source = ps_source.loc[row["charger_id"]]
    operators = row["plugshare_operators"]
    source_operator = clean_text(match_canonical_operator(source["operator"]))
    operator = source_operator if source_operator in operators else "; ".join(operators)
    adapter = {
        "ocm_id": row["plugshare_id"],
        "ocm_title": row["plugshare_name"],
        "ocm_street": row["plugshare_address"],
        "ocm_address": row["plugshare_address"],
        "ocm_operator": operator,
        "ocm_postcode": row["plugshare_postcode"],
        "ocm_latitude": row["plugshare_latitude"],
        "ocm_longitude": row["plugshare_longitude"],
        "ocm_dc_confirmed": row["dc_confirmed"],
    }
    if pd.isna(row["distance_m"]):
        evidence = {
            "candidate_passes": False,
            "rule_decision": "review",
            "candidate_reason": "missing/invalid coordinates",
        }
    else:
        # Compare street portions consistently on both sides; full addresses remain in the review.
        adapter["ocm_street"] = clean_text(row["plugshare_address"]).split(",")[0]
        evidence = match_compare(source, adapter, row["distance_m"])
    reason = evidence["candidate_reason"].replace("OCM", "PlugShare")
    ps_evidence.append(
        {
            **row,
            "station_address": source["station_address"],
            "operator": source["operator"],
            "postcode": source["postcode"],
            "plugshare_operator": "; ".join(operators),
            **{
                key: evidence.get(key)
                for key in [
                    "operator_relation",
                    "address_similar",
                    "postcode_match",
                    "postcode_conflict",
                    "postcode_missing_exception",
                    "candidate_passes",
                    "rule_decision",
                ]
            },
            "candidate_reason": reason,
        }
    )
ps_evidence = pd.DataFrame(ps_evidence)


## 2.6 Resolve PlugShare candidates

Select the nearest passing DC item. Exact distance ties use the external ID. One passing item
among AC-only alternatives can be accepted. Each source is matched independently: the same PlugShare ID can be accepted for multiple source records.
Unlike shared OCM sites, these are not removed here. Some are the same charger listed in two TfNSW
feeds, which Task 2 already flags (`probable_duplicate_flag`, Task 2 section 9.1); others may be two
units at one site. The cell after the review counts them.
Missing searches or failed searches stay unmatched and remain in the coverage denominator.


In [24]:
ps_rows = []
for source in ps_targets.to_dict("records"):
    candidates = (
        ps_evidence.loc[ps_evidence.charger_id.eq(source["charger_id"])]
        if not ps_evidence.empty
        else pd.DataFrame()
    )
    row = {
        "charger_id": source["charger_id"],
        "station_address": source["station_address"],
        "operator": source["operator"],
        "postcode": source["postcode"],
        "match_status": "unmatched",
        "match_reason": "no downloaded candidates",
        "search_has_dc_item": False,
    }
    if not candidates.empty:
        passing = candidates.loc[candidates.candidate_passes.eq(True)]
        chosen = (
            (passing if not passing.empty else candidates)
            .sort_values(["distance_m", "plugshare_id"])
            .iloc[0]
            .to_dict()
        )
        row.update(chosen)
        row["search_has_dc_item"] = bool(candidates.dc_confirmed.any())
        row["match_status"] = (
            "accepted" if len(passing) >= 1 else chosen["rule_decision"]
        )
        row["match_reason"] = chosen["candidate_reason"] + (
            f"; nearest of {len(passing)} passing candidates"
            if len(passing) > 1
            else ""
        )
        row["passing_candidates"] = len(passing)
    ps_rows.append(row)
ps_review = pd.DataFrame(ps_rows)
if "plugshare_id" not in ps_review:
    ps_review["plugshare_id"] = pd.NA
# Match each source independently; a PlugShare site may support several source records.
ps_review.to_csv(interim_dir / "plugshare_matching_review.csv", index=False)
ps_columns = [
    "charger_id",
    "station_address",
    "operator",
    "plugshare_address",
    "plugshare_operator",
    "distance_m",
    "postcode",
    "plugshare_postcode",
    "search_has_dc_item",
    "dc_confirmed",
    "dc_plug_types",
    "match_status",
    "match_reason",
]
print(ps_review.match_status.value_counts().to_string())
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    print("Accepted PlugShare matches:")
    display(
        ps_review.loc[ps_review.match_status.eq("accepted")]
        .reindex(columns=ps_columns)
        .head(5)
    )
    print("Unmatched / review:")
    display(
        ps_review.loc[~ps_review.match_status.eq("accepted")]
        .reindex(columns=ps_columns)
        .head(5)
    )


match_status
accepted     216
review        81
reject        11
unmatched      3
Accepted PlugShare matches:


,charger_id,station_address,operator,plugshare_address,plugshare_operator,distance_m,postcode,plugshare_postcode,search_has_dc_item,dc_confirmed,dc_plug_types,match_status,match_reason
0,2,"01 Wallgrove Road, Sydney, 2766",BP Australia,"1 Wallgrove Rd, Eastern Creek NSW 2766, Australia",BP Australia,62.845975,2766,2766,True,True,CCS2,accepted,50–250 m with same/aliased operator and similar street name; house number optional
3,7,"1 Bells Blvd, Kingscliff, 2487",Evie Networks,"Shop/3 Bells Blvd, Kingscliff NSW 2487, Australia",Evie Networks,6.644511,2487,2487,True,True,CCS2; CHAdeMO,accepted,within 50 m and same/aliased operator; address disagreement allowed
5,17,"1 Frederick St, Sydney, 2064",Evie Networks,"1 Frederick St, Artarmon NSW 2064, Australia",Evie Networks,48.048290,2064,2064,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed
6,20,"1 Greenbridge Dr, Wilton, 2571",Chargefox,"1 Greenbridge Dr, Wilton NSW 2571, Australia",Chargefox,3.253084,2571,2571,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed
8,37,"1 Perry St, Batemans Bay, 2536",Evie Networks,"15 Clyde St, Batemans Bay NSW 2536, Australia",Evie Networks,45.553781,2536,2536,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed


Unmatched / review:


,charger_id,station_address,operator,plugshare_address,plugshare_operator,distance_m,postcode,plugshare_postcode,search_has_dc_item,dc_confirmed,dc_plug_types,match_status,match_reason
1,3,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,"81 Marsh St, Armidale NSW 2350, Australia",NRMA,27.160117,2836,2350,True,True,CCS2; CHAdeMO,review,within 50 m and same/aliased operator; address disagreement allowed; postcodes disagree
2,6,"1 Bay St, Sydney, 2037",Tesla,"Basement/1 Bay St, Ultimo NSW 2007, Australia",,33.684068,2037,2007,True,False,,review,insufficient operator/address agreement for this distance
4,12,"1 Dalgal Way, Sydney, 2037",Tesla,"1 Dalgal Wy, Forest Lodge NSW 2037, Australia",Supercharger,20.095755,2037,2037,True,True,CCS2,review,different known operators
7,28,"1 Little Walker St, Casino, NSW 2470, Australia",NRMA,"15 Victory St, Braidwood NSW 2622, Australia",NRMA,5.370469,2470,2622,True,True,CCS2; CHAdeMO,review,within 50 m and same/aliased operator; address disagreement allowed; postcodes disagree
10,48,"1 Wharf St, Tweed Heads, 2485",Tesla,"Tweed Mall, corner, Wharf St, Tweed Heads NSW 2485, Australia",Supercharger,11.284035,2485,2485,True,True,CCS2,review,different known operators


In [25]:
# Several source records may accept the same PlugShare location. Both records are
# kept, as Task 2 keeps its probable duplicates: flagged there, not removed.
accepted_ps = ps_review.loc[ps_review.match_status.eq("accepted")]
shared_ps = accepted_ps.groupby("plugshare_id")["charger_id"].apply(sorted)
shared_ps = shared_ps[shared_ps.map(len) > 1]
flagged_ids = set(chargers.loc[chargers["probable_duplicate_flag"], "charger_id"])
already_flagged = shared_ps.map(lambda ids: bool(set(ids) & flagged_ids))
print(f"{len(shared_ps)} PlugShare locations were accepted for more than one source record; "
      f"{int(already_flagged.sum())} of these groups contain a record Task 2 flagged as a probable duplicate.")

14 PlugShare locations were accepted for more than one source record; 6 of these groups contain a record Task 2 flagged as a probable duplicate.


# 3. Final dataset

This section combines accepted attributes, validates the row set, and saves interim audit data and the compact processed CSV.


## 3.1 Combine plug types and save the final dataset

OCM matches take priority. PlugShare adds explicit DC plug types only to remaining records.
Normalise equivalent labels such as `CCS (Type 2)` to `CCS2`. The CSV uses semicolon-separated
unique plug names for portability; Task 4 can split these into a charger/plug-type relation.
Only non-empty plug types count toward plug-type coverage. Match IDs and retrieval timestamps
are saved with the final attributes. The review table retains rejected candidate information.


In [26]:
def standard_plugs(value):
    aliases = {"CCS (Type 2)": "CCS2", "CCS (Type 1)": "CCS1", "CHAdeMO": "CHAdeMO"}
    return "; ".join(
        sorted(
            {
                aliases.get(part.strip(), part.strip())
                for part in clean_text(value).split(";")
                if part.strip()
            }
        )
    )


### 3.2 Start with the deduplicated OCM records
Initialise the OCM plug type.


In [27]:
# Start with OCM-enriched records, then remove the OCM duplicate IDs selected earlier.
combined = augmented.loc[~augmented.charger_id.isin(dedup_remove_ids)].copy()
combined["ocm_plugtype"] = combined["ocm_plugtype"].map(standard_plugs)
combined["plug_type"] = combined["ocm_plugtype"]
combined["plug_type_source"] = np.where(
    combined.plug_type.notna() & combined.plug_type.ne(""), "Open Charge Map", ""
)


### 3.3 Prepare accepted PlugShare matches
Keep only PlugShare records that passed all matching rules.


In [28]:
ps_accepted = ps_review.loc[ps_review.match_status.eq("accepted")].copy()
assert ps_accepted.charger_id.is_unique  # One selected match per original record.
assert not set(ps_accepted.charger_id) & set(
    match_matches.loc[match_matches.match_status.eq("accepted"), "charger_id"]
)
ps_add = ps_accepted.reindex(
    columns=[
        "charger_id",
        "plugshare_id",
        "dc_plug_types",
        "plugshare_url",
        "retrieved_at",
    ]
).rename(
    columns={
        "dc_plug_types": "plugshare_plugtype",
        "retrieved_at": "plugshare_retrieved_at",
    }
)
combined = combined.merge(ps_add, on="charger_id", how="left", validate="one_to_one")
ps_has_plugs = combined.plugshare_plugtype.fillna("").ne("")
combined.loc[ps_has_plugs, "plug_type"] = combined.loc[
    ps_has_plugs, "plugshare_plugtype"
].map(standard_plugs)
combined.loc[ps_has_plugs, "plug_type_source"] = "PlugShare"


### 3.4 Set final match flags
Convert blank plug values to missing data and update PlugShare and overall augmentation status.


In [29]:
# Normalise the combined field before audit counts and final export.
combined["plug_type"] = combined["plug_type"].replace(
    r"^\s*$", pd.NA, regex=True
)
combined["plugshare_match_status"] = combined.charger_id.map(
    ps_review.set_index("charger_id").match_status
).fillna("not_targeted")
combined["overall_match_status"] = np.where(
    combined.plugshare_id.notna(), "accepted", combined.match_status
)
combined["augmented"] = combined.augmented | ps_has_plugs
assert not set(combined.charger_id) & set(dedup_remove_ids)
assert len(combined) == len(chargers) - len(dedup_remove_ids) and combined.charger_id.is_unique
expected_source = chargers.loc[
    ~chargers.charger_id.isin(dedup_remove_ids)
].copy()
pd.testing.assert_frame_equal(
    combined[chargers.columns].sort_values("charger_id").reset_index(drop=True),
    expected_source.sort_values("charger_id").reset_index(drop=True),
    check_dtype=False,
)

### 3.5 Save interim and final outputs
Write the complete audit file to `data/interim` and the compact dataset with one `plug_type` column to `data/processed`.


In [30]:
IMPORTANT_FINAL_COLUMNS = [
    'charger_id', 'site_id', 'station_name', 'station_address', 'lga_name', 'postcode',
    'latitude', 'longitude', 'sa4_code', 'sa4_name', 'gcc_code', 'gcc_name', 'state_name',
    'operator', 'charger_type', 'charger_status', 'number_of_plugs', 'power_kw_min',
    'power_kw_max', 'connectors_in_rating', 'rating_format', 'plug_type'
]

final_columns = [column for column in IMPORTANT_FINAL_COLUMNS if column in combined.columns]
# Select the public output columns and give the combined plug field its final name.
final_clean = combined[final_columns].copy()


In [31]:
# Save the compact final dataset; the full audit dataset is saved separately above.
final_clean.to_csv(processed_dir / 'ev_chargers_augmented.csv', index=False)
# Full audit dataset: every combined column, including OCM usage/pricing text,
# operator websites, external IDs and match statuses. Task 4 reads this file.
combined.to_csv(interim_dir / 'ev_chargers_augmented_full.csv', index=False)
plug_count = int((combined.charger_type.eq("DC") & combined.plug_type.notna() & combined.plug_type.ne("")).sum())
aug_count = int((combined.charger_type.eq("DC") & combined.augmented).sum())
print(f"Plug types: {plug_count}/{len(dc_chargers)} ({100*plug_count/len(dc_chargers):.2f}%)")
print(
    f"Any usable new attribute: {aug_count}/{len(dc_chargers)} ({100*aug_count/len(dc_chargers):.2f}%)"
)
# The six shared-OCM duplicates are no longer in the final dataset, so coverage
# is also stated against the DC records that remain.
final_dc = int(combined.charger_type.eq("DC").sum())
print(f"Against the {final_dc} DC records in the final dataset: "
      f"{aug_count}/{final_dc} ({100*aug_count/final_dc:.2f}%)")
print("Saved combined dataset: data/processed/ev_chargers_augmented.csv")


Plug types: 330/431 (76.57%)
Any usable new attribute: 330/431 (76.57%)
Against the 425 DC records in the final dataset: 330/425 (77.65%)
Saved combined dataset: data/processed/ev_chargers_augmented.csv


In [32]:
final_clean

,charger_id,site_id,station_name,station_address,lga_name,postcode,latitude,longitude,sa4_code,sa4_name,gcc_code,gcc_name,state_name,operator,charger_type,charger_status,number_of_plugs,power_kw_min,power_kw_max,connectors_in_rating,rating_format,plug_type
0,1,1,NaN,"Muswellbrook, 2333",Muswellbrook Shire Council,2333,-32.262242,150.890139,106,Hunter Valley exc Newcastle,1RNSW,Rest of NSW,New South Wales,EVUp,AC,Operational,2,22.0,22.0,NaN,number+unit,<NA>
1,2,2,NaN,"01 Wallgrove Road, Sydney, 2766",Blacktown City Council,2766,-33.811004,150.849597,116,Sydney - Blacktown,1GSYD,Greater Sydney,New South Wales,BP Australia,DC,Operational,4,150.0,150.0,NaN,number+unit,CCS2
2,3,3,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",Central Darling Shire Council,2836,-30.511874,151.669395,110,New England and North West,1RNSW,Rest of NSW,New South Wales,NRMA,DC,Operational,4,50.0,50.0,NaN,number+unit,<NA>
3,4,4,NaN,"1 Balfour St, Sydney, 2070",Ku-ring-gai Council,2070,-33.774101,151.167035,121,Sydney - North Sydney and Hornsby,1GSYD,Greater Sydney,New South Wales,Chargefox,AC,Operational,7,22.0,22.0,NaN,number+unit,<NA>
4,5,5,NaN,"1 Bay Ln, Byron Bay, 2481",Byron Shire Council,2481,-28.641819,153.613633,112,Richmond - Tweed,1RNSW,Rest of NSW,New South Wales,Tesla,AC,Operational,2,19.0,19.0,NaN,number+unit,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1935,1942,1931,NaN,17 Dulwich St Dulwich Hill NSW 2203 Austr...,Inner West Council,2203,-33.902552,151.140843,120,Sydney - Inner West,1GSYD,Greater Sydney,New South Wales,PLUS ES,AC,Operational,1,7.0,7.0,NaN,number+unit,<NA>
1936,1943,1932,NaN,17 Flood St Bondi NSW 2026 Australia,Waverley Council,2026,-33.891058,151.258787,118,Sydney - Eastern Suburbs,1GSYD,Greater Sydney,New South Wales,EVX,AC,Operational,2,22.0,22.0,NaN,number+unit,<NA>
1937,1944,1933,NaN,17 Grove St Dulwich Hill NSW 2203 Australia,Inner West Council,2203,-33.901739,151.139465,120,Sydney - Inner West,1GSYD,Greater Sydney,New South Wales,PLUS ES,AC,Operational,1,7.0,7.0,NaN,number+unit,<NA>
1938,1945,1934,NaN,17 Hereward St Maroubra NSW 2035 Australia,Randwick City Council,2035,-33.945104,151.256166,118,Sydney - Eastern Suburbs,1GSYD,Greater Sydney,New South Wales,PLUS ES,AC,Operational,1,22.0,22.0,NaN,number+unit,<NA>
